#### Purpose
Builds the baseline supervised model used as the main reference point in the project.
##### ⚙️The project follows this logic:
**raw data -> data quality -> feature engineering -> anomaly detection -> baseline model -> hybrid model -> performance benchmarking**

##### ⚙️Execution order
📝01_data_loading.ipynb -> 📝02_data_quality.ipynb -> 📝03_feature_engineering.ipynb -> 📝04_anomaly_detection.ipynb -> 
📝05_BaseLine.ipynb -> 📝06_HybridModel.ipynb -> 📝07_Interpretability_SHAP_Analysis.ipynb -> 📝08_performance_benchmarking.ipynb 

#### Main tasks
- load modeling features
- prepare train / validation / test logic
- train the baseline model
- evaluate predictive performance
- generate benchmark metrics

#### Output
- trained baseline model
- baseline metrics
- reference results for comparison

In [2]:
%load_ext autoreload
%autoreload 2

import os
import glob
from pathlib import Path
import sys
import pyarrow
import pandas as pd


PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_PROCESSED, DATA_FEATURES, MODELS_BASELINE,MODELS_METRICS
from src.repository.parquet_repository import ParquetRepository
from src.models.LightGBMRegressorBaseLine import LightGBMRegressorBaseLine
from src.models.LightGBMRegressorAnomaly import LightGBMRegressorAnomaly



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
repo = ParquetRepository(DATA_FEATURES)

In [4]:
df_load = repo.load("beverage_sales_feature.parquet")

[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\features\beverage_sales_feature.parquet


In [5]:
print (df_load.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

In [6]:
df_colunas_q = df_load.filter(regex='^Category')
print (df_colunas_q.dtypes)

Category    object
dtype: object


In [7]:
df_load["Order_Date"] = pd.to_datetime(df_load["Order_Date"])

df_train_baseline = df_load[df_load["Order_Date"].dt.year.isin([2021, 2022])].copy()
df_test_baseline = df_load[df_load["Order_Date"].dt.year == 2023].copy()

In [8]:
target_col = "quantity_sum"

numeric_features = [
    "total_price_sum",
    "unit_price_mean",
    "discount_mean",
    "order_count",
    "customer_count",
    "avg_ticket",
    "day_of_week",
    "month",
    "is_weekend",
    "quantity_sum_mean_7d",
    "quantity_sum_std_7d",
    "quantity_sum_sum_7d",
    "total_price_sum_mean_7d",
    "total_price_sum_std_7d",
    "total_price_sum_mean_30d",
    "unit_price_mean_mean_7d",
    "discount_mean_mean_14d",
    "quantity_vs_mean_7d",
    "total_price_vs_mean_30d",
    "quantity_pct_vs_mean_7d",
    "history_less_than_7d",
    "history_less_than_30d"
]

categorical_features = [
    "Product",
    "Region"
]

baseline_model = LightGBMRegressorBaseLine(
    target_col=target_col,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    model_dir=MODELS_BASELINE,
    random_state=3,
    n_splits=3,
    scoring="neg_root_mean_squared_error"
)

grid_result = baseline_model.fit(df_train_baseline)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


In [9]:
metrics = baseline_model.evaluate(df_train_baseline)

print(metrics)

d:\PROJETOS\git_repo\BEVERAGE-SALES\_venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


{'mae': 2.727535482124652, 'rmse': np.float64(4.1588411372652345), 'r2': 0.9993051320456422}


In [10]:
model_path = baseline_model.save_model()
params_path = baseline_model.save_best_params()
#metrics_path = baseline_model.save_metrics(MODELS_METRICS)

print("Model saved at:", model_path)
print("Best params saved at:", params_path)
#print("Metrics saved at:", metrics_path)

Model saved at: D:\PROJETOS\git_repo\BEVERAGE-SALES\models\baseline\lightgbm_baseline_model.joblib
Best params saved at: D:\PROJETOS\git_repo\BEVERAGE-SALES\models\baseline\lightgbm_baseline_best_params.json
